# Machine Learning on Human Carbonic Anhydrase II Inhibition



**Author:** [Babak Mamnoon/GitHub]
**Project:** Classification of hCA II Inhibitors using ChEMBL data and ECFP4 Fingerprints.

### Installing Cheminformatics and ML libraries

In [ ]:

!pip install rdkit chembl_webresource_client xgboost -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from chembl_webresource_client.new_client import new_client
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import (roc_curve, auc, precision_recall_curve,
                             confusion_matrix, average_precision_score, accuracy_score)

print("Libraries imported successfully.")

### Data Acquisition & Exploration

In [ ]:
print("--- Step A: Extracting ChEMBL Dataset ---")

TARGET_ID = 'CHEMBL205'
activity = new_client.activity
res = activity.filter(target_chembl_id=TARGET_ID).filter(standard_type="IC50")

df_raw = pd.DataFrame.from_dict(res)
df = df_raw.dropna(subset=['standard_value', 'canonical_smiles']).copy()
df['standard_value'] = pd.to_numeric(df['standard_value'], errors='coerce')
df = df.dropna(subset=['standard_value'])

df['class'] = df['standard_value'].apply(lambda x: 1 if x < 1000 else 0)

print(f"Total entries retrieved: {len(df_raw)}")
print(f"Cleaned entries (valid SMILES/IC50): {len(df)}")
print("-" * 30)
display(df[['molecule_chembl_id', 'canonical_smiles', 'standard_value', 'class']].head())

### Molecular Featurization (ECFP4)

In [ ]:
print("--- Step B: Converting SMILES to Morgan Fingerprints ---")

def generate_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        return np.array(AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048))
    return None

df['fp'] = df['canonical_smiles'].apply(generate_fp)
df = df.dropna(subset=['fp'])

X = np.stack(df['fp'].values)
y = df['class'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

### Model Training & Evaluation

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
    "SVM (RBF)": SVC(kernel='rbf', probability=True, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

performance_log = {}

for name, model in models.items():
    print(f"Processing {name}...")
    model.fit(X_train, y_train)
    y_probs = model.predict_proba(X_test)[:, 1]
    y_preds = model.predict(X_test)

    fpr, tpr, _ = roc_curve(y_test, y_probs)
    roc_auc = auc(fpr, tpr)
    performance_log[name] = {"ROC-AUC": roc_auc, "Accuracy": accuracy_score(y_test, y_preds)}

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(fpr, tpr, label=f'AUC = {roc_auc:.3f}')
    ax[0].set_title(f'{name} ROC')
    ax[0].legend()

    cm = confusion_matrix(y_test, y_preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=ax[1])
    ax[1].set_title(f'{name} Confusion Matrix')
    plt.show()

### Final Comparison

In [ ]:
sorted_summary = sorted(performance_log.items(), key=lambda x: x[1]['ROC-AUC'], reverse=True)
print("\nFINAL SCIENTIFIC CONCLUSION")
for i, (name, stats) in enumerate(sorted_summary):
    print(f"{i+1}. {name}: ROC-AUC = {stats['ROC-AUC']:.3f}, Accuracy = {stats['Accuracy']:.1%}")